# Gymnasium: BipedalWalker-v3

Our objective is to train an agent to navigate the BipedalWalker environment using Reinforcement Learning. Before implementing complex algorithms or aiming for advanced maneuvers (like doing a flip), we need to understand the environment's dynamics.

## Environment Overview
`BipedalWalker-v3` is a 2D physics simulation environment from the Gymnasium Box2D environments. The goal is to make a bipedal robot walk to the right end of the terrain. In the `hardcore=True` version, the terrain is not flat; it includes obstacles such as ladders, stumps, and pitfalls.

### Action Space
The action space is a continuous `Box(-1.0, 1.0, (4,), float32)`. The agent controls the robot by applying torques to its four main joints. The four values in the action array represent:
1. Hip 1 (Torque / Speed)
2. Knee 1 (Torque / Speed)
3. Hip 2 (Torque / Speed)
4. Knee 2 (Torque / Speed)

All action values must be within the `[-1.0, 1.0]` range.

### Reward System
The agent receives rewards based on its forward progress and energy efficiency. According to the official documentation, the reward is calculated as follows:
* **Forward Movement:** The agent is rewarded for moving forward (to the right). Reaching the end of the terrain yields a total of over 300 points.
* **Falling Penalty:** If the robot's main body (hull) touches the ground, it falls. This results in a heavy penalty of **-100** points, and the episode terminates immediately.
* **Motor Torque Penalty:** To encourage efficient, natural walking rather than chaotic flailing, applying motor torque costs a small negative reward.
### Set up the Environment

In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import gymnasium as gym
env = gym.make("BipedalWalker-v3", hardcore=False, render_mode="human") # Human we can see the environment

## Baseline: Random Actions

To establish a baseline and visualize how an untrained agent interacts with the physics engine, we will run a single episode using a random policy. The agent will sample actions uniformly from the action space until the episode ends.

An episode ends if:
- `terminated` is True (the agent falls or reaches the goal).
- `truncated` is True (the agent runs out of time/steps).
- Past 20 seconds of simulation time.

In [21]:
import time

env = gym.make("BipedalWalker-v3", hardcore=False, render_mode="human") 
limit_time = 10  # seconds
start_time = time.time()
obs, info = env.reset()

terminated = False
truncated = False
total_reward = 0.0
step_count = 0

# Loop until the agent finishes or fails
while not (terminated or truncated) and (time.time() - start_time < limit_time):
    # Sample a random continuous action within [-1, 1] for the 4 joints
    action = env.action_space.sample() 
    
    # Step the environment forward
    obs, reward, terminated, truncated, info = env.step(action)
    
    total_reward += reward
    step_count += 1

# Close the rendering window
env.close()

print(f"Episode finished after {step_count} steps.")
print(f"Total Reward with random policy: {total_reward:.2f}")

Episode finished after 52 steps.
Total Reward with random policy: -105.10


### Changing the Environment
Instead of training the agent only to walk, we can reshape the task so it learns to perform a flip.
The main idea is to change the reward and encourage trunk rotation, airtime, and landing control.


Changes in the robot:
- Track the robot body's orientation.
- Reward angular velocity and rotation progress.
- Give a large bonus when the agent completes a full rotation.
- Reduce the fall penalty so the agent is willing to take risks.
- Penalize forward movement less, so the policy focuses on flipping rather than walking.

In [2]:
import gymnasium as gym
import numpy as np
import custom_bipedal
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor

class CurriculumFlipperWrapper(gym.Wrapper):
    def __init__(self, env, stage=1, max_steps=1500):
        super().__init__(env)
        self.max_steps = max_steps
        self.stage = stage 
        
        # Initialize tracking variables (will be reset in reset())
        self.cumulative_angle = 0.0
        self.prev_angle = 0.0
        self.flip_completed = False
        self.step_counter = 0
        
        # Expand observation space to include continuous absolute angle progress
        low = np.append(self.env.observation_space.low, -np.inf)
        high = np.append(self.env.observation_space.high, np.inf)
        self.observation_space = gym.spaces.Box(low, high, dtype=np.float32)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.step_counter += 1

        current_angle = obs[0]
        delta_angle = current_angle - self.prev_angle

        # Handle wrap-around
        if delta_angle > np.pi:
            delta_angle -= 2 * np.pi
        elif delta_angle < -np.pi:
            delta_angle += 2 * np.pi

        prev_cumulative = self.cumulative_angle
        self.cumulative_angle += delta_angle
        self.prev_angle = current_angle

        # Use absolute values so Backflips are equally rewarded!
        abs_angle = abs(self.cumulative_angle)
        abs_prev = abs(prev_cumulative)

        custom_reward = 0.0

        # 1. Base Rotation Rewards
        custom_reward += abs(obs[1]) * 5.0  # Reward raw rotational speed in either direction
        
        # Because we use abs(), wiggling back and forth yields negative progress!
        rotation_progress = abs_angle - abs_prev
        custom_reward += rotation_progress * 15.0 

        # Airtime bonus
        if obs[8] == 0.0 and obs[13] == 0.0:
            custom_reward += 1.0

        # Intermediate milestones
        milestones = [np.pi / 2, np.pi, 3 * np.pi / 2]
        for milestone in milestones:
            attr = f"_milestone_{milestone:.2f}_done"
            if not getattr(self, attr, False) and abs_angle >= milestone:
                setattr(self, attr, True)
                custom_reward += 50.0

        # 2. Flip Completion
        if abs_angle >= 2 * np.pi and not self.flip_completed:
            self.flip_completed = True
            custom_reward += 300.0 

        # 3. Stage-Specific Logic
        is_falling = (reward == -100)
        
        if self.stage == 1:
            if is_falling:
                custom_reward += 100.0 
                
        elif self.stage == 2:
            if self.flip_completed:
                upright_bonus = np.cos(obs[0]) 
                custom_reward += max(0, upright_bonus) * 3.0 
                custom_reward -= abs(obs[1]) * 3.0 
                
                y_vel = obs[3] 
                if y_vel < 0:
                    custom_reward += y_vel * 5.0 
                
                if obs[8] == 1.0 and obs[13] == 1.0:
                    custom_reward += 15.0 
            
            if is_falling and not self.flip_completed:
                custom_reward += 50.0

        if self.step_counter >= self.max_steps:
            truncated = True

        info['flip_completed'] = self.flip_completed
        info['cumulative_angle'] = self.cumulative_angle

        total_reward = reward + custom_reward

        # Feed the absolute angle percentage to the observation network
        obs = np.append(obs, abs_angle / (2 * np.pi)).astype(np.float32)
        return obs, total_reward, terminated, truncated, info

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        
        # Lower gravity in the custom environment unwrapped world
        self.env.unwrapped.world.gravity = (0.0, -5.0)

        self.cumulative_angle = 0.0
        self.prev_angle = obs[0]
        self.flip_completed = False
        self.step_counter = 0
        for milestone in [np.pi / 2, np.pi, 3 * np.pi / 2]:
            setattr(self, f"_milestone_{milestone:.2f}_done", False)
            
        obs = np.append(obs, 0.0).astype(np.float32)
        return obs, info

def make_env(stage):
    def _init():
        # Instantiate from custom_bipedal
        env = custom_bipedal.BipedalWalker()
        env = CurriculumFlipperWrapper(env, stage=stage)
        return env
    return _init


## Integrating Stable Baselines 3

Using Stable Baselines 3, we can implement a Proximal Policy Optimization (PPO) agent to learn how to flip in the BipedalWalker environment.

In [17]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv

# --- STAGE 1: ROTATION ---
def make_env_stage1():
    def _init():
        e = custom_bipedal.BipedalWalker(hardcore=False)
        return CurriculumFlipperWrapper(e, stage=1)
    return _init

vec_env_s1 = SubprocVecEnv([make_env_stage1() for _ in range(32)])

model = PPO("MlpPolicy", vec_env_s1, verbose=1, device="cuda", 
            n_steps=2048, batch_size=256, learning_rate=3e-4,
            tensorboard_log="./ppo_bipedal_curriculum/")

print("Starting Stage 1: Rotation Mastery...")
model.learn(total_timesteps=5_000_000) 
model.save("ppo_bipedal_stage1")
vec_env_s1.close()

# --- STAGE 2: LANDING ---
def make_env_stage2():
    def _init():
        e = custom_bipedal.BipedalWalker(hardcore=False)
        return CurriculumFlipperWrapper(e, stage=2)
    return _init

vec_env_s2 = SubprocVecEnv([make_env_stage2() for _ in range(32)])

print("Starting Stage 2: Stabilization and Landing...")
# Load the weights from Stage 1 into the new environment
model = PPO.load("ppo_bipedal_stage1", env=vec_env_s2, device="cuda")

model.learning_rate = 5e-5 # Changed to 5e-5

model.learn(total_timesteps=4_000_000) 
model.save("ppo_bipedal_final")


Using cuda device
Starting Stage 1: Rotation Mastery...
Logging to ./ppo_bipedal_curriculum/PPO_1
------------------------------
| time/              |       |
|    fps             | 4840  |
|    iterations      | 1     |
|    time_elapsed    | 13    |
|    total_timesteps | 65536 |
------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 2683        |
|    iterations           | 2           |
|    time_elapsed         | 48          |
|    total_timesteps      | 131072      |
| train/                  |             |
|    approx_kl            | 0.005123549 |
|    clip_fraction        | 0.0497      |
|    clip_range           | 0.2         |
|    entropy_loss         | -5.65       |
|    explained_variance   | 0.00147     |
|    learning_rate        | 0.0003      |
|    loss                 | 35          |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0039     |
|    std     

### Testing Model


In [ ]:
import time
import os
import gymnasium as gym
import torch
from stable_baselines3 import PPO

# 1. Corrected model path to match what was saved at the end of Stage 2
model_path = "ppo_bipedal_final.zip" 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Create a single human-render environment and wrap it
import custom_bipedal
env = custom_bipedal.BipedalWalker(hardcore=False, render_mode="human")

# 2. Corrected Wrapper name and explicitly set to Stage 2
env = CurriculumFlipperWrapper(env, stage=2)

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model not found: {model_path}. Train and save the model before running tests.")

print(f"Loading model from {model_path}...")
model = PPO.load(model_path, device=device)

num_episodes = 5
max_seconds = 15

for ep in range(1, num_episodes + 1):
    obs, info = env.reset()
    start_time = time.time()
    done = False
    total_reward = 0.0
    step_count = 0

    print(f"\n=== Test Episode {ep} ===")
    while not done and (time.time() - start_time) < max_seconds:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
        step_count += 1

    flip_completed = info.get("flip_completed", False)
    final_angle = info.get("cumulative_angle", 0.0)

    print(f"Episode {ep} — steps: {step_count}, total_reward: {total_reward:.2f}")
    print(f"Flip completed: {flip_completed}, Final cumulative rotation: {final_angle:.2f} radians")

    time.sleep(0.5)

env.close()


Using device: cuda
Loading model from ppo_bipedal_final.zip...


c:\Users\franc\Documents\FEUP\MEST\1ANO\2SEM\ASMA\ASMA2\venv\Lib\site-packages\gymnasium\spaces\box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
c:\Users\franc\Documents\FEUP\MEST\1ANO\2SEM\ASMA\ASMA2\venv\Lib\site-packages\gymnasium\spaces\box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
c:\Users\franc\Documents\FEUP\MEST\1ANO\2SEM\ASMA\ASMA2\venv\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and th


=== Test Episode 1 ===
Episode 1 — steps: 222, total_reward: 1024.48
Flip completed: True, Final cumulative rotation: 5.74 radians

=== Test Episode 2 ===
Episode 2 — steps: 217, total_reward: 925.89
Flip completed: True, Final cumulative rotation: 5.73 radians

=== Test Episode 3 ===
Episode 3 — steps: 737, total_reward: 9285.57
Flip completed: True, Final cumulative rotation: 7.12 radians

=== Test Episode 4 ===
Episode 4 — steps: 220, total_reward: 1013.09
Flip completed: True, Final cumulative rotation: 5.74 radians

=== Test Episode 5 ===
Episode 5 — steps: 216, total_reward: 936.52
Flip completed: True, Final cumulative rotation: 5.73 radians
